# 1일차 · 내 배터리는 몇 살인가
**충북대학교 · Python 활용 머신러닝 및 딥러닝 특강** 

이 노트북은 **위에서부터 한 셀씩 실행**합니다. 셀 왼쪽의 ▶ 를 누르거나 `Shift + Enter`.
코드를 타이핑할 일은 없습니다. `# ←` 표시가 있는 **숫자와 이름만** 바꿉니다.

| 막히면 | 이렇게 |
|---|---|
| 빨간 오류가 뜸 | 오류 아래 **「오류 설명」** 버튼 → 그래도 모르면 오류 글을 복사해 Zoom 채팅에 |
| 어디까지 했는지 모르겠음 | 메뉴 **런타임 → 모두 실행** 뒤 맨 아래부터 거슬러 보기 |
| ✅ 표시 | 결과 숫자를 **Zoom 채팅**에 올립니다(진도 확인용) |

먼저 **파일 → Drive에 사본 저장**을 눌러 내 것으로 만드세요.

## 블록 1 · 첫 실행과 내 배터리

### 0. Colab이 되는지 확인
이 셀이 하는 일 — 글자 한 줄을 화면에 찍습니다. 아래에 `안녕, 배터리!`가 보이면 환경은 끝.

In [ ]:
print("안녕, 배터리!")

✅ **체크포인트 ①** — 채팅에 `1` 을 올려 주세요(실행 성공).

### 1. 내 노트북 배터리 건강 상태(SOH) 계산
Windows: `Win + R` → `cmd` → `powercfg /batteryreport` → `C:\Users\내이름\battery-report.html` 열기 →
**DESIGN CAPACITY**(설계 용량)와 **FULL CHARGE CAPACITY**(지금 최대 용량)를 아래에 적습니다. 단위(mWh)는 그대로.
Mac: 시스템 정보 → 전원 → 「최대 용량 %」가 곧 SOH.

이 셀이 하는 일 — 두 숫자를 나눠 백분율로 만듭니다. **SOH = 지금 최대 용량 ÷ 설계 용량 × 100**

In [ ]:
design = 50000   # ← 내 보고서의 DESIGN CAPACITY (mWh)
full   = 42000   # ← 내 보고서의 FULL CHARGE CAPACITY (mWh)

soh = full / design * 100
print(f"내 노트북 배터리 SOH: {soh:.1f} %")

✅ **체크포인트 ②** — 내 SOH 숫자(예: `84.0`)를 채팅에 올려 주세요. 보고서가 안 만들어지면 `design = 50000, full = 42000` 그대로 실행.

### 2. 우리 반 배터리 분포 (강사가 채팅 숫자를 모아 실행)
이 셀이 하는 일 — 숫자 목록을 **히스토그램**(몇 % 구간에 몇 명)으로 그립니다.

In [ ]:
import matplotlib.pyplot as plt

soh_list = [84.0, 91.5, 78.2, 95.0, 88.7, 99.1, 73.4, 90.0]   # ← 채팅에 올라온 SOH들

plt.hist(soh_list, bins=range(60, 105, 5), edgecolor="white")
plt.xlabel("SOH (%)"); plt.ylabel("students"); plt.title("Our class laptop batteries")
plt.show()
print(f"평균 {sum(soh_list)/len(soh_list):.1f} %  ·  최저 {min(soh_list)}  ·  최고 {max(soh_list)}")

## 블록 2 · 파이썬 읽는 법과 진짜 배터리 데이터

### 3. 파이썬 읽기 ① 변수 · 리스트 · 반복
이 셀이 하는 일 — 용량 5개를 **리스트**에 담고, **for**로 하나씩 꺼내 SOH로 바꿔 찍습니다.
- `capacities = [...]` 이름표를 붙여 보관 = **변수**
- `[ , , ]` 여러 값을 순서대로 = **리스트**
- `for c in capacities:` 리스트에서 하나씩 꺼내 `c`라고 부름 = **반복**

In [ ]:
capacities = [1.86, 1.85, 1.84, 1.60, 1.40]   # Ah (정격 2.0 Ah)

for c in capacities:
    soh = c / 2.0 * 100
    print(c, "Ah  →  SOH", round(soh), "%")

### 4. 파이썬 읽기 ② 함수 부르기
이 셀이 하는 일 — 이미 만들어진 도구(**함수**)를 부릅니다. `len()` 개수, `max()` 최대, `min()` 최소. 괄호 안에 재료를 넣으면 결과가 나옵니다.

In [ ]:
print("몇 개:", len(capacities))
print("가장 큼:", max(capacities), " 가장 작음:", min(capacities))
print("평균:", round(sum(capacities) / len(capacities), 3))

### 5. NASA 배터리 데이터 읽기
이 셀이 하는 일 — 인터넷의 CSV 파일을 **표(DataFrame)** 로 읽어 옵니다. `pandas`는 표를 다루는 도구 모음.
데이터: NASA가 18650 리튬이온 셀을 상온에서 충방전 반복해 늙힌 기록. **방전 1회 = 1줄.**

In [ ]:
import pandas as pd

BASE = "https://raw.githubusercontent.com/aicec/battery-ml-data/main/"
summary = pd.read_csv(BASE + "cycles_summary.csv")

summary.head()     # 앞 5줄 보기

열 뜻 — `cell_id` 셀 이름 · `cycle` 방전 회차 · **`capacity_Ah` 방전 용량** · `soh_pct` 용량÷2.0×100 · `discharge_time_s` 방전 시간 · `max_temp` 방전 중 최고 온도 · `re_ohm` `rct_ohm` 내부 저항

이 셀이 하는 일 — 표의 크기와 셀 종류를 확인합니다.

In [ ]:
print("행(줄) 수:", len(summary))
print("셀 종류:", summary.cell_id.unique().tolist())
print("셀마다 몇 회:", summary.cell_id.value_counts().to_dict())

✅ **체크포인트 ③** — `행 수`를 채팅에 올려 주세요.

### 6. 표 다루기 — 셀 하나만 골라내기
이 셀이 하는 일 — `cell_id`가 `"B0005"`인 줄만 골라 `b5`라는 새 표를 만듭니다. `표[조건]` 꼴.

In [ ]:
b5 = summary[summary.cell_id == "B0005"]

print(len(b5), "회")
b5[["cycle", "capacity_Ah", "soh_pct", "max_temp"]].head(10)

### 7. 용량 곡선 — 늙어 가는 모습
이 셀이 하는 일 — x축 사이클, y축 용량으로 **선 그래프**를 그리고, 80%(1.6 Ah) 선을 빨간 점선으로 긋습니다.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(b5.cycle, b5.capacity_Ah, marker=".", linewidth=1)
plt.axhline(1.6, color="red", linestyle="--", label="80 % (1.6 Ah)")
plt.xlabel("cycle"); plt.ylabel("capacity (Ah)"); plt.title("B0005 capacity fade")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

✅ **체크포인트 ④** — 그래프를 캡처해 PAD 게시판 1일차에 올려 주세요(수업 끝까지 올리면 됩니다).

**눈으로 읽기** — 빨간 선 아래로 **처음** 내려가는 사이클이 몇일까요? 중간에 용량이 **거꾸로 오르는** 자리도 보이나요?

## 블록 3 · 셀 비교 · 80% 지점 · 방전 곡선

### 8. 셀 3개 겹쳐 그리기
이 셀이 하는 일 — `groupby`로 셀별 묶음을 만들고, 묶음마다 선을 하나씩 그립니다.

In [ ]:
plt.figure(figsize=(8, 4))
for cell, g in summary.groupby("cell_id"):
    plt.plot(g.cycle, g.capacity_Ah, label=cell)
plt.axhline(1.6, color="red", linestyle="--", label="80 %")
plt.xlabel("cycle"); plt.ylabel("capacity (Ah)"); plt.title("Three cells, same test")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

같은 실험인데 늙는 속도가 다릅니다. **어느 셀이 제일 먼저 80%에 닿나요?**

### 9. 80% 지점을 코드로 찾기
이 셀이 하는 일 — 용량이 1.6 이하인 줄만 골라, 그중 **첫 줄**의 사이클 번호를 찍습니다.

In [ ]:
below = b5[b5.capacity_Ah <= 1.6]
first = below.cycle.iloc[0]          # 첫 줄의 cycle 값

print("B0005가 80% 아래로 처음 떨어진 사이클:", first)
print("그때 용량:", below.capacity_Ah.iloc[0], "Ah")

✅ **체크포인트 ⑤** — 그 사이클 번호를 채팅에 올려 주세요.

### 10. 방전 곡선 — 젊은 셀 vs 늙은 셀
이 셀이 하는 일 — 방전 **중** 기록(초 단위 전압)을 읽어, 1회차와 100회차의 전압 곡선을 겹칩니다.

In [ ]:
curves = pd.read_csv(BASE + "discharge_curves.csv")

young = curves[(curves.cell_id == "B0005") & (curves.cycle == 1)]
old   = curves[(curves.cell_id == "B0005") & (curves.cycle == 100)]

plt.figure(figsize=(8, 4))
plt.plot(young.t_s / 60, young.voltage, label="cycle 1 (young)")
plt.plot(old.t_s / 60,   old.voltage,   label="cycle 100 (old)")
plt.xlabel("minutes"); plt.ylabel("voltage (V)"); plt.title("B0005 discharge curve")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

**읽기** — 늙은 셀은 전압이 **더 빨리** 떨어져 방전이 **더 일찍** 끝납니다. 같은 2 A로 뽑는데 시간이 짧다 = 담긴 전기가 적다 = **용량이 작다**. 그래서 방전 시간만 알아도 용량을 거의 알 수 있습니다(2일차에 이 얘기를 다시 합니다).

### 11. 오늘의 결정 문장
숫자를 **결정**으로 바꿔 봅니다. 아래 문장의 빈칸을 채워 채팅에 올려 주세요.

> 「B0005는 사이클 **___** 에서 보증 한계(80%)에 닿는다. 보증이 100사이클이라면 이 셀은 **(출하 가능 / 출하 불가)**.」

## 바꿔 보기 미션 (40분)
셀을 **복사하지 말고**, `# ←` 자리만 바꿔 실행합니다. 결과는 캡처해서 PAD 게시판 1일차에.

### 미션 1 · 다른 셀의 80% 사이클
9번 셀을 B0006, B0007로 바꿔 각각 몇 사이클인지 찾으세요.

In [ ]:
cell = "B0005"   # ← "B0006" 으로, 다음엔 "B0007" 로

g = summary[summary.cell_id == cell]
below = g[g.capacity_Ah <= 1.6]
print(cell, "→ 80% 아래 첫 사이클:", below.cycle.iloc[0])

### 미션 2 · 온도도 늙을까
7번 셀의 y축을 용량 대신 **방전 중 최고 온도**(`max_temp`)로 바꿔 그리세요. 사이클이 갈수록 온도가 어떻게 변하나요?

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(b5.cycle, b5.capacity_Ah, color="orange")   # ← capacity_Ah 를 max_temp 로
plt.xlabel("cycle"); plt.ylabel("max temperature (C)"); plt.title("B0005 discharge max temp")
plt.grid(alpha=0.3); plt.show()

### 미션 3 (도전) · 방전 곡선 4장 겹치기
10번 셀을 고쳐 1 · 50 · 100 · 160회차 네 곡선을 한 그래프에 그리세요. 힌트: `for n in [1, 50, 100, 160]:` 로 반복.

In [ ]:
plt.figure(figsize=(8, 4))
for n in [1, 100]:                      # ← [1, 50, 100, 160] 으로
    c = curves[(curves.cell_id == "B0005") & (curves.cycle == n)]
    plt.plot(c.t_s / 60, c.voltage, label=f"cycle {n}")
plt.xlabel("minutes"); plt.ylabel("voltage (V)"); plt.title("B0005 aging")
plt.legend(); plt.grid(alpha=0.3); plt.show()

## 오늘 배운 것 · 내일
- 배터리 **SOH = 지금 용량 ÷ 설계 용량**. 내 노트북도, NASA 실험 셀도 같은 식.
- 파이썬은 **읽을 줄 알면** 된다 — 변수 · 리스트 · 반복 · 함수 · 표.
- 데이터는 매끈하지 않다. 그래도 **80% 지점**은 코드 두 줄로 찾는다.
- **내일** — 80%가 되는 때를 **미리** 아는 법(머신러닝). 오늘 그래프의 앞부분만 보고 뒷부분을 맞힙니다.

### 과제
미션 1~3 결과 캡처를 PAD 게시판 1일차에 올리기. 내 노트북 SOH도 함께.